In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -U ultralytics opencv-python deep-sort-realtime


In [ ]:
from google.colab import drive
from google.colab import files
import shutil
import datetime
import cv2
from ultralytics import YOLOWorld
from deep_sort_realtime.deepsort_tracker import DeepSort

uploaded = files.upload()
video_filename = list(uploaded.keys())[0]

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

drive_input_path = f"/content/drive/MyDrive/input_{timestamp}_{video_filename}"
drive_output_path = f"/content/drive/MyDrive/output_DeepSORT_{timestamp}.mp4"

shutil.copy(video_filename, drive_input_path)


def process_video_with_yolo_deepsort(video_path, output_path, skip_frames=2):
    # Load YOLOWorld model
    model = YOLOWorld('/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt')

    # Initialize DeepSORT tracker
    tracker = DeepSort(max_age=10)

    # Open video
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    prev_tracks = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        if frame_count % skip_frames == 0:
            # Run YOLOWorld inference on every Nth frame
            results = model.predict(frame, conf=0.25, iou=0.6, augment=False, verbose=False)
            results = results[0]


            detections = []
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                conf = float(box.conf[0].cpu().numpy())
                cls = int(box.cls[0].cpu().numpy())
                detections.append(([x1, y1, x2 - x1, y2 - y1], conf, cls))

            # Update DeepSORT tracker
            tracks = tracker.update_tracks(detections, frame=frame)
            prev_tracks = tracks  # Save latest valid tracks
        else:
            # On skipped frames, reuse last known tracks (optional visualization only)
            tracks = prev_tracks

        # Draw tracking boxes
        for track in tracks:
            if not track.is_confirmed():
                continue
            l, t, r, b = map(int, track.to_ltrb())
            track_id = track.track_id
            cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {track_id}", (l, t - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()



In [ ]:
process_video_with_yolo_deepsort(video_path=video_filename, output_path=drive_output_path)

print(" Output saved to:", drive_output_path)